# Job Radar - Data Analyst (InHire)

Este notebook busca vagas automaticamente em empresas que usam a plataforma InHire.

Funcionalidades:

- Busca vagas em múltiplas empresas
- Filtra vagas de dados
- Filtra vagas remotas
- Mostra resultados no terminal
- Preparado para enviar alertas no Telegram


In [ ]:
# =============================
# IMPORTS
# =============================

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re


## Configurações

In [ ]:
# =============================
# EMPRESAS QUE USAM INHIRE
# =============================

COMPANIES = [
    "sympla",
    "contabilizei",
    "quintoandar",
    "kiwify",
    "loft",
    "vtex",
    "hotmart",
    "olist"
]


# =============================
# PALAVRAS-CHAVE DE VAGAS
# =============================

KEYWORDS = [
    "data",
    "analytics",
    "analyst",
    "bi",
    "machine learning",
    "cientista",
    "dados"
]


# =============================
# PALAVRAS DE REMOTO
# =============================

REMOTE_KEYWORDS = [
    "remote",
    "remoto",
    "anywhere",
    "brazil"
]


# =============================
# TELEGRAM (opcional)
# =============================

TELEGRAM_TOKEN = "COLOQUE_SEU_TOKEN_AQUI"
TELEGRAM_CHAT_ID = "COLOQUE_SEU_CHAT_ID_AQUI"


## Função para coletar vagas

In [ ]:
# =============================
# SCRAPER DE VAGAS
# =============================

def fetch_jobs(company):

    url = f"https://{company}.inhire.app/vagas"

    jobs = []

    try:

        response = requests.get(url, timeout=10)

        if response.status_code != 200:
            return jobs

        soup = BeautifulSoup(response.text, "html.parser")

        links = soup.find_all("a")

        for link in links:

            href = link.get("href")
            text = link.text.strip()

            if not href:
                continue

            if "vaga" in href or "job" in href:

                jobs.append({
                    "empresa": company,
                    "titulo": text,
                    "link": f"https://{company}.inhire.app{href}"
                })

    except Exception as e:

        print("Erro em", company, e)

    return jobs


## Filtro de vagas de dados

In [ ]:
# =============================
# FILTRAR VAGAS DE DADOS
# =============================

def is_data_job(title):

    title = title.lower()

    for k in KEYWORDS:
        if k in title:
            return True

    return False


## Envio para Telegram

In [ ]:
# =============================
# ENVIAR ALERTA TELEGRAM
# =============================

def send_telegram(message):

    if TELEGRAM_TOKEN == "COLOQUE_SEU_TOKEN_AQUI":
        return

    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"

    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": message
    }

    requests.post(url, json=payload)


## Executar Radar

In [ ]:
# =============================
# EXECUÇÃO DO RADAR
# =============================

all_jobs = []

for company in COMPANIES:

    print("Escaneando:", company)

    jobs = fetch_jobs(company)

    all_jobs.extend(jobs)

    time.sleep(1)

print("\nTotal vagas encontradas:", len(all_jobs))


# =============================
# FILTRAR VAGAS DATA
# =============================

data_jobs = [job for job in all_jobs if is_data_job(job['titulo'])]

print("Vagas de dados encontradas:", len(data_jobs))


# =============================
# MOSTRAR RESULTADOS
# =============================

for job in data_jobs:

    message = f"""
🚨 Nova vaga de Dados

Empresa: {job['empresa']}
Cargo: {job['titulo']}
Link: {job['link']}
"""

    print(message)

    send_telegram(message)


# =============================
# SALVAR CSV
# =============================

df = pd.DataFrame(data_jobs)

df.to_csv("vagas_data_analyst.csv", index=False)

print("\nArquivo salvo: vagas_data_analyst.csv")
